# 23. Preprocessing Test (Pipeline 04)

- Goal: inspect monocular pose-quality preprocessing on the default recording with the selected exercise definition.
- Docs: `docs_eng/pipeline/04_preprocessing.md` / `docs/pipeline/04_preprocessing.md`
- Inputs: Annotated pose dataframe and selected exercise-definition YAML.
- Outputs: In-memory preprocessed dataframe/report unless a cell explicitly saves under `data/processed/`.
- Checks: direct preprocessing output, QC summary, configuration provenance, synthetic far-side diagnostic, and pipeline integration.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import json
import warnings

import pandas as pd

from movement.config import LANDMARKS
from movement.pipeline import (
    FarSideStabilizationConfig,
    InterpolationConfig,
    PreprocessingConfig,
    ReliabilityConfig,
    SmoothingConfig,
    run_pipeline,
)
from movement.preprocessing import preprocess_pose_dataframe
from movement.stage_context import (
    build_stage_check_pipeline_config,
    prepare_previous_stage_inputs,
)

print('imports OK')


## Data Setup

Runs the prior context in pipeline order before preprocessing:

```text
Pose CSV -> ① Validation -> ② Annotation -> ③ Exercise Definition -> ④ Preprocessing
```


In [ ]:
pose_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv'
annotation_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv'
TARGET_EXERCISE_ID = 'squat'

stage_inputs = prepare_previous_stage_inputs(
    prepare_until='exercise_definition',
    pose_csv=pose_csv,
    annotation_csv=annotation_csv,
    exercise_id=TARGET_EXERCISE_ID,
    landmarks=LANDMARKS,
)

df_raw = stage_inputs.raw_df
ann_df = stage_inputs.annotation_df
df_annotated = stage_inputs.annotated_df
val_report = stage_inputs.validation_report
ann_report = stage_inputs.annotation_report
exercise_def = stage_inputs.exercise_definition
TARGET_DEFINITIONS_DIR = stage_inputs.definitions_dir

setup_summary = pd.DataFrame([
    {'item': 'frames_loaded', 'value': len(df_raw)},
    {'item': 'validation_passed', 'value': val_report['passed']},
    {'item': 'structural_validation_passed', 'value': val_report.get('structural_passed')},
    {'item': 'analysis_frames', 'value': f"{ann_report['num_analysis_frames']} / {ann_report['num_total_frames']}"},
    {'item': 'exercise_id', 'value': exercise_def.exercise_id},
    {'item': 'movement_template_id', 'value': exercise_def.classification['movement_template_id']},
    {'item': 'laterality', 'value': exercise_def.classification['laterality']},
    {'item': 'definitions_dir', 'value': str(TARGET_DEFINITIONS_DIR)},
])
display(setup_summary)

if val_report.get('warnings'):
    display(pd.DataFrame(val_report['warnings']))
if not val_report['passed']:
    print('NOTE: structural validation failed; inspect before preprocessing.')


## Direct Preprocessing Test

Call `preprocess_pose_dataframe()` directly for a focused stage-level check.


In [ ]:
pre_config = PreprocessingConfig(
    enabled=True,
    reliability=ReliabilityConfig(
        confidence_threshold=0.5,
        segment_length_tolerance=0.25,
        joint_angle_check=True,
        velocity_threshold_torso_per_sec=5.0,
    ),
    interpolation=InterpolationConfig(enabled=True, method='linear', max_gap_frames=3),
    smoothing=SmoothingConfig(enabled=False),
)

pre_df, pre_report = preprocess_pose_dataframe(
    df=df_annotated,
    landmarks=LANDMARKS,
    exercise_definition=exercise_def,
    config=pre_config,
)

added_columns = [c for c in pre_df.columns if c not in df_annotated.columns]
print(f'preprocessed shape: {pre_df.shape}')
print(f'added columns: {len(added_columns)}')
print(json.dumps({k: pre_report[k] for k in ["method", "exercise_id", "movement_template_id", "execution_pattern", "num_invalid_frames"]}, indent=2, ensure_ascii=False))


## Check 1: Output Columns and Report Contract


In [ ]:
required_frame_columns = ['preprocessing_valid', 'preprocessing_note', 'swap_corrected']
missing_frame_columns = [col for col in required_frame_columns if col not in pre_df.columns]
assert not missing_frame_columns, f'missing frame-level columns: {missing_frame_columns}'

present_landmarks = [lm for lm in LANDMARKS if f'{lm}_x' in pre_df.columns]
missing_split_columns = [
    col
    for lm in present_landmarks
    for col in (
        f'{lm}_observed_reliable',
        f'{lm}_usable',
        f'{lm}_preprocessing_source',
    )
    if col not in pre_df.columns
]
assert not missing_split_columns, f'missing observed/usable/source columns: {missing_split_columns}'

required_report_keys = [
    'method', 'exercise_id', 'movement_template_id', 'execution_pattern', 'laterality',
    'num_frames', 'num_coordinate_columns', 'reliability_summary',
    'landmark_quality_summary', 'rule_contribution_summary',
    'worst_landmarks_by_observed_unreliable', 'worst_landmarks_by_unusable',
    'frames_with_many_unusable_landmarks', 'swap_detection_summary',
    'interpolation_summary', 'smoothing_summary', 'num_invalid_frames',
    'applied_columns',
]
missing_report_keys = [key for key in required_report_keys if key not in pre_report]
assert not missing_report_keys, f'missing report keys: {missing_report_keys}'
assert pre_report['exercise_id'] == TARGET_EXERCISE_ID
assert pre_report['movement_template_id'] == exercise_def.classification['movement_template_id']

contract_summary = pd.DataFrame([
    {'item': 'frame_level_columns_present', 'value': True},
    {'item': 'landmarks_with_split_reliability_columns', 'value': len(present_landmarks)},
    {'item': 'required_report_keys_present', 'value': True},
    {'item': 'swap_detection_enabled', 'value': pre_report['swap_detection_summary']['enabled']},
    {'item': 'swap_corrected_frames', 'value': pre_report['swap_detection_summary']['num_temporal_swap_corrected']},
    {'item': 'smoothing_enabled', 'value': pre_report['smoothing_summary']['enabled']},
])
display(contract_summary)
print('PASS: preprocessing output contract is valid')


## Check 2: QC Summary and Configuration


In [ ]:
rel = pre_report['reliability_summary']
itp = pre_report['interpolation_summary']
target_landmarks = [lm for lm in LANDMARKS if f'{lm}_x' in pre_df.columns]
total_landmark_frames = max(len(pre_df) * len(target_landmarks), 1)
valid_frame_ratio = (len(pre_df) - pre_report['num_invalid_frames']) / max(len(pre_df), 1)
observed_unreliable_ratio = rel['num_observed_unreliable_landmark_frames'] / total_landmark_frames
unusable_ratio = rel['num_unusable_landmark_frames'] / total_landmark_frames
interpolation_attempted = (
    itp['num_landmark_frames_recovered']
    + itp['num_post_velocity_rejected_landmark_frames']
)
interpolation_recovery_ratio = (
    itp['num_landmark_frames_recovered'] / max(interpolation_attempted, 1)
)
post_velocity_rejection_ratio = (
    itp['num_post_velocity_rejected_landmark_frames']
    / max(interpolation_attempted, 1)
)


def _names(rows, key, limit=5):
    return ', '.join(str(row[key]) for row in rows[:limit]) or 'none'


def _ratio(value):
    return round(float(value), 4)


if post_velocity_rejection_ratio > 0 or unusable_ratio >= 0.20:
    qc_interpretation = 'review_recommended'
elif observed_unreliable_ratio > 0 or unusable_ratio > 0:
    qc_interpretation = 'ready_with_low_confidence_notes'
else:
    qc_interpretation = 'ready_for_next_stage'

stage_qc_summary = pd.DataFrame([
    {'item': 'exercise_id', 'value': pre_report['exercise_id']},
    {'item': 'movement_template_id', 'value': pre_report['movement_template_id']},
    {'item': 'execution_pattern', 'value': pre_report.get('execution_pattern')},
    {'item': 'qc_interpretation', 'value': qc_interpretation},
    {'item': 'preprocessing_valid_frames', 'value': f"{len(pre_df) - pre_report['num_invalid_frames']} / {len(pre_df)}"},
    {'item': 'preprocessing_valid_frame_ratio', 'value': _ratio(valid_frame_ratio)},
    {'item': 'observed_unreliable_landmark_ratio', 'value': _ratio(observed_unreliable_ratio)},
    {'item': 'unusable_landmark_ratio', 'value': _ratio(unusable_ratio)},
    {'item': 'interpolated_landmark_frames_recovered', 'value': itp['num_landmark_frames_recovered']},
    {'item': 'post_velocity_rejected_landmark_frames', 'value': itp['num_post_velocity_rejected_landmark_frames']},
    {'item': 'top_observed_unreliable_landmarks', 'value': _names(pre_report['worst_landmarks_by_observed_unreliable'], 'landmark')},
    {'item': 'top_unusable_landmarks', 'value': _names(pre_report['worst_landmarks_by_unusable'], 'landmark')},
    {'item': 'frames_with_many_unusable_landmarks', 'value': _names(pre_report['frames_with_many_unusable_landmarks'], 'frame')},
])
display(stage_qc_summary)

active_preprocessing_config = pd.DataFrame([
    {'area': 'reliability', 'setting': 'confidence_threshold', 'value': pre_config.reliability.confidence_threshold},
    {'area': 'reliability', 'setting': 'segment_length_tolerance', 'value': pre_config.reliability.segment_length_tolerance},
    {'area': 'reliability', 'setting': 'joint_angle_check', 'value': pre_config.reliability.joint_angle_check},
    {'area': 'reliability', 'setting': 'velocity_threshold_torso_per_sec', 'value': pre_config.reliability.velocity_threshold_torso_per_sec},
    {'area': 'interpolation', 'setting': 'config_max_gap_frames', 'value': pre_config.interpolation.max_gap_frames},
    {'area': 'interpolation', 'setting': 'effective_max_gap_frames', 'value': exercise_def.quality_rules.max_interpolation_gap_frames},
    {'area': 'interpolation', 'setting': 'post_velocity_check', 'value': pre_config.interpolation.post_velocity_check},
    {'area': 'smoothing', 'setting': 'enabled', 'value': pre_config.smoothing.enabled},
])
display(active_preprocessing_config)
print('PASS: preprocessing QC/config summary generated')


## Check 3: Far-Side Observation Confidence Diagnostic

Synthetic diagnostic only; these counts are not target-recording movement-quality results.


In [ ]:
side_view_df = df_annotated.copy()
side_view_df['camera_zone'] = 'Z3'

for lm in ['left_shoulder', 'left_hip', 'left_knee', 'left_ankle']:
    col = f'{lm}_z'
    if col in side_view_df.columns:
        side_view_df[col] = -0.2
for lm in ['right_shoulder', 'right_hip', 'right_knee', 'right_ankle']:
    col = f'{lm}_z'
    if col in side_view_df.columns:
        side_view_df[col] = 0.2

jitter_frame = len(side_view_df) // 2
side_view_df.loc[jitter_frame, 'right_knee_x'] = side_view_df.loc[jitter_frame, 'right_knee_x'] + 1.0
if 'right_knee_confidence' in side_view_df.columns:
    side_view_df.loc[jitter_frame, 'right_knee_confidence'] = 0.2

far_side_config = PreprocessingConfig(
    enabled=True,
    reliability=ReliabilityConfig(
        confidence_threshold=0.5,
        segment_length_tolerance=0.25,
        joint_angle_check=True,
        velocity_threshold_torso_per_sec=5.0,
    ),
    interpolation=InterpolationConfig(enabled=False),
    smoothing=SmoothingConfig(enabled=False),
    far_side_stabilization=FarSideStabilizationConfig(
        enabled=True,
        max_gap_frames=1,
    ),
)

far_pre_df, far_pre_report = preprocess_pose_dataframe(
    df=side_view_df,
    landmarks=LANDMARKS,
    exercise_definition=exercise_def,
    config=far_side_config,
)

far_summary = far_pre_report['far_side_stabilization_summary']
availability = far_pre_report['feature_availability_summary']

diagnostic_summary = pd.DataFrame([
    {
        'observed_zones': ', '.join(far_summary['camera_side_inference']['observed_zones']),
        'observed_high_jitter_far_side_frames': far_summary['num_observed_high_jitter_far_side_landmark_frames'],
        'post_high_jitter_far_side_frames': far_summary['num_post_preprocessing_high_jitter_far_side_landmark_frames'],
        'jitter_policy': far_summary['jitter_detection_policy'],
        'symmetry_gate_ready': availability['symmetry_gate_ready'],
        'low_confidence_features': ', '.join(availability['low_confidence_feature_families']),
    }
])
display(diagnostic_summary)
assert far_summary['enabled'] is True
print('NOTE: synthetic perturbation diagnostic only; do not interpret these counts as target-recording movement quality.')


## Check 4: Pipeline Integration


In [ ]:
pipe_config = build_stage_check_pipeline_config(
    exercise_id=TARGET_EXERCISE_ID,
    definitions_dir=TARGET_DEFINITIONS_DIR,
    enable_annotation=False,
    preprocessing_config=PreprocessingConfig(enabled=True),
)

with warnings.catch_warnings(record=True):
    warnings.simplefilter("always")
    pipe_df, pipe_report = run_pipeline(
        df_annotated,
        config=pipe_config,
        landmarks=LANDMARKS,
    )

assert "preprocessing" in pipe_report
assert pipe_report["exercise_definition"]["exercise_id"] == TARGET_EXERCISE_ID

pipeline_summary = pd.DataFrame([
    {'item': 'steps_executed', 'value': list(pipe_report.keys())},
    {'item': 'pipeline_exercise_id', 'value': pipe_report['exercise_definition']['exercise_id']},
    {'item': 'pipeline_output_shape', 'value': pipe_df.shape},
    {'item': 'pipeline_preprocessing_invalid_frames', 'value': pipe_report['preprocessing']['num_invalid_frames']},
])
display(pipeline_summary)
print('PASS: preprocessing step in pipeline report')


## Check Summary

This notebook is a compact execution/QC checkpoint for ④ Preprocessing. The QC/config tables are provenance for stage review, not biomarker scores.
